# Darkroom: run the web app from Colab

Before you start: Runtime > Change runtime type > **T4 GPU**. Put `server.py` and the `static` folder in Drive at `MyDrive/darkroom_app`. Run both cells; the link stays alive while this tab is open. Anyone with the link can use your GPU, so don't share it.

In [ ]:
!pip install -q fastapi uvicorn diffusers transformers accelerate peft safetensors
!pip uninstall -y -q torchao
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared

In [ ]:
import os, re, subprocess, sys, time, urllib.request
from google.colab import drive
drive.mount("/content/drive")

APP  = "/content/drive/MyDrive/darkroom_app"    # server.py and static/ go here
LORA = "/content/drive/MyDrive/lora_general/output/pytorch_lora_weights.safetensors"
assert os.path.exists(f"{APP}/server.py"), "Upload server.py and the static folder to MyDrive/darkroom_app"
assert os.path.exists(LORA), "LoRA file not found in Drive"

log = open("/content/server.log", "w")
srv = subprocess.Popen([sys.executable, "-m", "uvicorn", "server:app", "--host", "127.0.0.1", "--port", "8000"],
                       cwd=APP, env={**os.environ, "LORA_PATH": LORA}, stdout=log, stderr=subprocess.STDOUT)
for _ in range(300):    # first start downloads SDXL (~7 GB), so this can take several minutes
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/api/health", timeout=2); break
    except Exception:
        if srv.poll() is not None: raise RuntimeError(open("/content/server.log").read()[-2000:])
        time.sleep(3)
else:
    raise RuntimeError("Server did not start in time. See /content/server.log")

tun = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
                       stdout=subprocess.DEVNULL, stderr=open("/content/tunnel.log", "w"))
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.log").read())
    if m: print("Open this link:", m.group(0)); break
    time.sleep(1)
else:
    print("No link yet. See /content/tunnel.log")